# Impulse — A2D2 Analysis: safety event search + event statistics + clips

Companion to `a2d2_ingestion.ipynb` and `a2d2_object_detection.ipynb`. It performs **event
search** over the silver-layer channels — fusing **bus signals** (speed, braking, steering)
with the **object-detection channels** (`detected_pedestrians` / `detected_cyclists` /
`detected_vehicles`) to find **safety-relevant scenarios** — computes **statistics within
those events**, and exports each event as an H.264 MP4 clip onto the volume.

## Prerequisite
Run `a2d2_ingestion.ipynb` (with `download_images=true`) and `a2d2_object_detection.ipynb`
first. The detection notebook adds the `detected_*` channels; without them only the
bus-signal events are evaluated (the perception-fused events are skipped automatically).

## License
A2D2 © Audi AG, **CC BY-ND 4.0** (https://creativecommons.org/licenses/by-nd/4.0/, source
https://www.a2d2.audi/). This notebook reads only your own ingested tables/images — it
redistributes nothing. Commit it with outputs cleared.

In [ ]:
%pip install pydantic>=2.0 scipy pillow imageio imageio-ffmpeg -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.text("clip_max_duration_s", "10", "Clip max duration (s)")
dbutils.widgets.text("clip_fps", "5", "Clip playback fps")
dbutils.widgets.text("clips_per_event", "5", "Clips per event name")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
CLIP_MAX_DURATION_S = float(dbutils.widgets.get("clip_max_duration_s") or "10")
CLIP_FPS = float(dbutils.widgets.get("clip_fps") or "5")
CLIPS_PER_EVENT = int(dbutils.widgets.get("clips_per_event") or "5")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

# Works from a cloned repo/Git folder (src on path) and as a DABs wheel-installed job.
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
# (multi-container: the engine computes events/stats per container_id automatically; no single
# CONTAINER_ID is assumed here.)
REPORT_NAME = "a2d2_analysis"
vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
clips_dir = f"{vol_root}/{REPORT_NAME}"   # event clips written here (folder = report name)
if not spark.catalog.tableExists(f"{pfx}_channels"):
    raise RuntimeError(f"{pfx}_channels not found. Run a2d2_ingestion.ipynb first.")
print("Reading from", f"{pfx}_*", "| clips ->", clips_dir)

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.sequence_of_events import SequenceOfEvents
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
import numpy as np
import pandas as pd
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name=REPORT_NAME, spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()

## 1. Channels — bus signals + perception channels

Select the bus channels used by the events, and the object-detection channels added by
`a2d2_object_detection.ipynb` (if present).

In [ ]:
speed = db.query.channel(channel_name="vehicle_speed")
acc_x = db.query.channel(channel_name="acceleration_x")
steer = db.query.channel(channel_name="steering_angle_calculated")
brake = db.query.channel(channel_name="brake_pressure")

# Perception channels exist only if a2d2_object_detection.ipynb has run.
chan_names = set(r["value"] for r in spark.read.table(f"{pfx}_channel_tags")
                 .where("key = 'channel_name'").select("value").distinct().collect())
have_counts = any(n.startswith("detected_") for n in chan_names)
have_distance = "pedestrian_nearest_distance_m" in chan_names and "vehicle_nearest_distance_m" in chan_names
print("perception:", "distance+counts" if have_distance else ("counts only" if have_counts else
      "NONE - run a2d2_object_detection.ipynb (with estimate_distance=true) for perception events"))
if have_counts:
    ped = db.query.channel(channel_name="detected_pedestrians")
    cyc = db.query.channel(channel_name="detected_cyclists")
    veh = db.query.channel(channel_name="detected_vehicles")
have_ahead = "vehicle_ahead_nearest_distance_m" in chan_names
if have_distance:
    ped_dist = db.query.channel(channel_name="pedestrian_nearest_distance_m")
    cyc_dist = db.query.channel(channel_name="cyclist_nearest_distance_m")
    veh_dist = db.query.channel(channel_name="vehicle_nearest_distance_m")
# Center-ahead (in-path) distances, when the detection step produced them: the lead vehicle /
# pedestrian-in-path, filtering out oncoming/parked/cross-traffic. Falls back to nearest-any.
if have_ahead:
    veh_ahead = db.query.channel(channel_name="vehicle_ahead_nearest_distance_m")
    ped_ahead = db.query.channel(channel_name="pedestrian_ahead_nearest_distance_m")

## 2. Safety event search (proximity × dynamics)

Interesting scenes come from the **rare tail** of the distributions, not the average — so
thresholds are calibrated to this drive's tail (slow urban drive, max ~38 km/h: decel min
~-3.7 m/s², brake p99 ~44 bar, steering mean only 34° but p99 ~371°, closest pedestrian ~6 m,
closest lead vehicle ~4.6 m). Each event combines **proximity × dynamics** at tail thresholds:

- **`emergency_braking`** — `acc_x < -3.0 m/s²` or `brake > 45 bar`, **while moving (> 5 km/h)**
  (near the hardest stops; excludes standstill brake-pedal-holding at lights).
- **`sharp_cornering`** — `|steering| > 200°` while `speed > 15 km/h` (a genuinely hard/fast
  turn, well above the 34° mean).
- **`evasive_maneuver`** — `SequenceOfEvents`: emergency braking *then* sharp steering
  (brake-then-swerve collision avoidance).
- **`pedestrian_near_miss`** — pedestrian `< 9 m` (closest decile) while moving `> 15 km/h`.
- **`imminent_rear_end`** — time-headway `< 2.5 s` to the **in-path lead vehicle** while
  `> 10 km/h` (`thw = vehicle_ahead_nearest_distance_m / (vehicle_speed/3.6)`). Uses the
  center-ahead vehicle channel, so oncoming/parked/cross-traffic no longer leak in (falls back to
  nearest-any vehicle if the `ahead_*` channels aren't present).
- **`pedestrian_in_path`** — a pedestrian **center-ahead** (in the ego path) within
  `PED_PATH_M` (any speed; the highest-criticality VRU scenario; needs `*_ahead_*` channels).
- **`cyclist_close_encounter`** — cyclist `< 12 m` while moving (cyclists are rare → notable).
- **`vru_brake_reaction`** — `SequenceOfEvents`: a close VRU appears, *then* emergency braking.

(If distance channels aren't available, it falls back to count-based VRU-while-braking; if no
perception at all, only the bus-signal maneuvers run.)

In [ ]:
EVENT_PAD_S = 2.5
PAD_US = int(EVENT_PAD_S * 1e6)   # .expand() works in the series' time unit (RAW = microseconds)

# Thresholds calibrated to the TAIL of this drive's distributions (decel min ~-3.7, brake p99
# ~44, steering mean 34 / p99 371 [unsigned], pedestrian p10 ~8.8m, vehicle p10 ~6.4m). Tunable.
EMERG_DECEL, EMERG_BRAKE, EMERG_MIN_KMH = -3.0, 45.0, 5.0
SHARP_STEER_DEG, CORNER_KMH = 200.0, 15.0
PED_NEAR_M, PED_MOVE_KMH = 9.0, 15.0
# imminent_rear_end uses the in-path lead vehicle (vehicle_ahead_*), so oncoming/parked cars are
# gone at the source; thw<2.5s surfaces the closest genuine lead-vehicle following on this calm drive.
REAREND_THW_S, REAREND_KMH = 2.5, 10.0
PED_PATH_M = 15.0                          # pedestrian directly in the ego path (any speed)
CYC_NEAR_M, CYC_MOVE_KMH = 12.0, 8.0
VRU_REACT_M = 10.0

# Bus-signal events (steering_angle_calculated is unsigned: min 0). Emergency braking requires
# the car to be MOVING (speed > EMERG_MIN_KMH) so standstill brake-pedal-holding is excluded.
emergency_brake = (((acc_x < EMERG_DECEL) | (brake > EMERG_BRAKE)) & (speed > EMERG_MIN_KMH)).expand(PAD_US)
sharp_steer = (steer > SHARP_STEER_DEG)
ev_brake = BasicEvent(name="emergency_braking", expr=emergency_brake,
                      desc=f"accel_x<{EMERG_DECEL} m/s^2 or brake>{EMERG_BRAKE:.0f} bar, while moving >{EMERG_MIN_KMH:.0f} km/h (+/-2.5s)")
ev_corner = BasicEvent(name="sharp_cornering", expr=((sharp_steer) & (speed > CORNER_KMH)).expand(PAD_US),
                       desc=f"|steering|>{SHARP_STEER_DEG:.0f} deg while >{CORNER_KMH:.0f} km/h")
# Evasive maneuver: emergency braking FOLLOWED BY sharp steering (brake-then-swerve).
ev_evasive = SequenceOfEvents(name="evasive_maneuver", expressions=[emergency_brake, sharp_steer.expand(PAD_US)],
                              desc="emergency braking then sharp (evasive) steering")
events = [ev_brake, ev_corner, ev_evasive]

if have_distance:
    # proximity x dynamics at the rare tail — genuinely interesting, not mere presence.
    # Use the in-path lead vehicle (vehicle_ahead_*) for time-headway when available, else nearest-any.
    lead_dist = veh_ahead if have_ahead else veh_dist
    thw = lead_dist / (speed / 3.6)                                     # time-headway, seconds
    vru_near = (ped_dist < VRU_REACT_M) | (cyc_dist < CYC_NEAR_M)        # close VRU (for the reaction sequence)
    events += [
        BasicEvent(name="pedestrian_near_miss",
                   expr=((ped_dist < PED_NEAR_M) & (speed > PED_MOVE_KMH)).expand(PAD_US),
                   desc=f"pedestrian <{PED_NEAR_M:.0f}m while moving >{PED_MOVE_KMH:.0f} km/h"),
        BasicEvent(name="imminent_rear_end",
                   expr=((thw < REAREND_THW_S) & (speed > REAREND_KMH)).expand(PAD_US),
                   desc=f"time-headway <{REAREND_THW_S}s to the in-path lead vehicle while >{REAREND_KMH:.0f} km/h"),
        BasicEvent(name="cyclist_close_encounter",
                   expr=((cyc_dist < CYC_NEAR_M) & (speed > CYC_MOVE_KMH)).expand(PAD_US),
                   desc=f"cyclist <{CYC_NEAR_M:.0f}m while moving >{CYC_MOVE_KMH:.0f} km/h"),
        SequenceOfEvents(name="vru_brake_reaction", expressions=[vru_near.expand(PAD_US), emergency_brake],
                   desc="close VRU, then an emergency braking reaction (ordered)"),
    ]
    if have_ahead:
        # pedestrian directly in the ego path while moving — the highest-criticality VRU scenario
        events.append(BasicEvent(name="pedestrian_in_path",
                   expr=(ped_ahead < PED_PATH_M).expand(PAD_US),
                   desc=f"pedestrian in path (center-ahead) <{PED_PATH_M:.0f}m (any speed)"))
elif have_counts:
    # fallback (no distance): count-based co-occurrence, still not presence-only
    vru = ((ped > 0) | (cyc > 0)).expand(PAD_US)
    events += [
        BasicEvent(name="vru_while_braking", expr=vru & emergency_brake, desc="VRU visible during emergency braking"),
        SequenceOfEvents(name="vru_then_braking", expressions=[vru, emergency_brake], desc="VRU visible, then braking"),
    ]

for e in events:
    report.add_event(e)
print("registered events:", [e.get_name() for e in report.get_events()])

## 3. Statistics within events

Compute per-instance statistics **within every event** (one `StatsAggregator` per event) — speed,
deceleration, brake pressure, steering, and (when available) the nearest / in-path pedestrian &
vehicle distances during each event instance — so the app can show the safety context of *any*
event, not just braking. A final aggregation computes **whole-drive** statistics for the
**center-ahead (in-path) channels** (`event=None`) to verify those channels carry real signal.

In [ ]:
stats_inputs = [speed, acc_x, brake, steer]
stats_names = ["vehicle_speed", "acceleration_x", "brake_pressure", "steering_angle_calculated"]
if have_distance:
    stats_inputs += [ped_dist, veh_dist]
    stats_names += ["pedestrian_nearest_distance_m", "vehicle_nearest_distance_m"]
elif have_counts:
    stats_inputs += [ped, veh]
    stats_names += ["detected_pedestrians", "detected_vehicles"]
if have_ahead:
    stats_inputs += [veh_ahead, ped_ahead]
    stats_names += ["vehicle_ahead_nearest_distance_m", "pedestrian_ahead_nearest_distance_m"]

# Within-event statistics for EVERY defined event (one StatsAggregator per event), so the app can
# show stats for any selected event instance -- not just emergency_braking. The engine tags each
# stats_aggregator_fact row with its event_id/event_instance_id.
page = Page(page_number=1)
for e in events:
    page.add_aggregation(StatsAggregator(
        name=f"{e.get_name()}_stats",
        input_expressions=stats_inputs,
        channel_names=stats_names,
        statistics=["min", "mean", "max"],
        event=e,
    ))

# Whole-drive statistics for the center-ahead (in-path) channels (event=None) -- shows whether
# the ahead_* channels carry real signal across the whole drive, not just within events.
ahead_names = sorted(n for n in chan_names if "ahead" in n)
if ahead_names:
    page.add_aggregation(StatsAggregator(
        name="ahead_channel_stats",
        input_expressions=[db.query.channel(channel_name=n) for n in ahead_names],
        channel_names=ahead_names,
        statistics=["min", "mean", "max"],
        event=None,
    ))
    print("ahead channels included in stats:", ahead_names)

report.add_page(page)
report.determine_report()
report.persist_results()

print("Event instances found per event:")
display(spark.read.table(f"{pfx}_event_instance_fact")
        .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id")
        .groupBy("event_name").count().orderBy(F.desc("count")))

print("Event instances per city x event (container_tags.city):")
_city = (spark.read.table(f"{pfx}_container_tags").where("key = 'city'")
         .select("container_id", F.col("value").alias("city")))
display(spark.read.table(f"{pfx}_event_instance_fact")
        .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id")
        .join(_city, "container_id", "left")
        .groupBy("city", "event_name").count().orderBy("city", F.desc("count")))

print("Within-event statistics per event (averaged over instances):")
display(spark.read.table(f"{pfx}_stats_aggregator_fact")
        .where("event_id IS NOT NULL")
        .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id")
        .groupBy("event_name", "channel_name", "aggregation_label")
        .agg(F.round(F.avg("statistic_value"), 3).alias("avg_over_instances"))
        .orderBy("event_name", "channel_name", "aggregation_label"))

if ahead_names:
    print("Whole-drive statistics for the center-ahead (in-path) channels:")
    display(spark.read.table(f"{pfx}_stats_aggregator_fact")
            .where("channel_name LIKE '%ahead%'")
            .groupBy("channel_name", "aggregation_label")
            .agg(F.round(F.avg("statistic_value"), 3).alias("value"))
            .orderBy("channel_name", "aggregation_label"))

## 4. Export event video clips (MP4)

For **every event name**, take its first `clips_per_event` instances, and for each instance
encode the camera frames within the event's own window
(`[start_ts, min(end_ts, start_ts+clip_max_duration_s)]` — events are `.expand()`-ed so the
window already spans a useful range) into an **H.264 MP4** on the volume. The file name
includes the event name:
`{vol_root}/<report_name>/<event_name>_<container_id>_<event_instance_id>.mp4`.

In [ ]:
import shutil
import imageio.v2 as imageio
from PIL import Image

def write_mp4(image_paths, out_path, fps=5.0, width=640):
    """Encode image files into an H.264 MP4 at out_path (must be a local, seekable path)."""
    frames = []
    for p in image_paths:
        im = Image.open(p).convert("RGB")
        w, h = im.size
        nw = width - (width % 2)                 # H.264 needs even dimensions
        nh = int(round(h * nw / w)); nh -= nh % 2
        im = im.resize((nw, max(2, nh)))
        frames.append(np.asarray(im))
    if not frames:
        raise ValueError("no frames")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    imageio.mimwrite(out_path, frames, fps=fps, codec="libx264", quality=8, macro_block_size=1)
    return out_path

def event_clip_frames(frames_pdf, container_id, start_ts, end_ts, max_duration_s=5.0, fps=5.0, max_frames=300):
    """Frames of THIS container within the event window [start_ts, min(end_ts, start_ts+
    max_duration_s)] (us), temporally downsampled to ~fps frames/sec by keeping one frame per
    1/fps-second bucket. Filtering by container_id keeps clips to the event's own drive (multi-
    container). Downsampling to `fps` and writing at the same `fps` plays the clip at REAL TIME."""
    end = min(int(end_ts), int(start_ts) + int(max_duration_s * 1e6))
    sub = (frames_pdf[(frames_pdf.container_id == container_id)
                      & (frames_pdf.cam_tstamp_us >= start_ts) & (frames_pdf.cam_tstamp_us <= end)]
           .sort_values("cam_tstamp_us"))
    if len(sub) > 1 and fps and fps > 0:
        bucket = max(1, int(1e6 / fps))
        sub = sub.assign(_b=(sub.cam_tstamp_us // bucket)).drop_duplicates("_b").drop(columns="_b")
    if len(sub) > max_frames:
        idx = np.linspace(0, len(sub) - 1, max_frames).round().astype(int)
        sub = sub.iloc[idx]
    return sub

def write_event_clip(event_name, container_id, instance_id, sub_pdf, clips_dir, fps=5.0, width=640):
    """Encode frames to a local mp4 (seekable), then copy to the volume folder clips_dir.
    File name = <event_name>_<container_id>_<event_instance_id>.mp4."""
    fname = f"{event_name}_{int(container_id)}_{int(instance_id)}.mp4"
    tmp = f"/tmp/a2d2_clips/{fname}"
    write_mp4(sub_pdf.png_path.tolist(), tmp, fps=fps, width=width)
    os.makedirs(clips_dir, exist_ok=True)
    dst = f"{clips_dir}/{fname}"
    shutil.copyfile(tmp, dst)                    # finished file -> FUSE volume (no seek needed)
    return dst

In [ ]:
from pyspark.sql.window import Window
if not spark.catalog.tableExists(f"{pfx}_camera_frames"):
    print(f"{pfx}_camera_frames not found - re-run ingestion with download_images=true.")
else:
    frames_pdf = (spark.read.table(f"{pfx}_camera_frames")
                  .select("container_id", "cam_tstamp_us", "png_path").orderBy("cam_tstamp_us").toPandas())
    ev = (spark.read.table(f"{pfx}_event_instance_fact")
          .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id"))
    # up to CLIPS_PER_EVENT clips per (event_name, container) so every drive's events are shown
    _w = Window.partitionBy("event_name", "container_id").orderBy("start_ts")
    ranked = (ev.withColumn("_rn", F.row_number().over(_w)).where(F.col("_rn") <= CLIPS_PER_EVENT)
              .select("event_name", "container_id", "event_instance_id", "start_ts", "end_ts")
              .orderBy("event_name", "container_id", "start_ts").toPandas())
    if ranked.empty or frames_pdf.empty:
        print("No event instances or no camera frames available.")
    else:
        total = 0
        for _, r in ranked.iterrows():
            sub = event_clip_frames(frames_pdf, r["container_id"], r["start_ts"], r["end_ts"],
                                    CLIP_MAX_DURATION_S, CLIP_FPS)
            if len(sub) == 0:
                continue
            write_event_clip(r["event_name"], r["container_id"], r["event_instance_id"], sub, clips_dir, fps=CLIP_FPS)
            total += 1
        print(f"Wrote {total} MP4 clip(s) under {clips_dir} "
              f"(<event_name>_<container_id>_<event_instance_id>.mp4, up to {CLIPS_PER_EVENT} per event x container):")
        display(spark.createDataFrame(dbutils.fs.ls(clips_dir)).orderBy("name"))

## Notes
- This notebook persists **event** facts (`_event_instance_fact`, `_event_dimension`) and
  **statistics** (`_stats_aggregator_fact`/`_dimension`) — no histograms.
- Perception-fused events (`vru_while_braking`, `vru_then_braking`, …) require the
  `detected_*` channels from `a2d2_object_detection.ipynb`; without them only the bus-signal
  events run.
- Event clips are MP4 (H.264) on the volume under `{vol_root}/<report_name>/`, named
  `<event_name>_<container_id>_<event_instance_id>.mp4` — up to `clips_per_event` per event
  name. Clips play at **real time**: window frames are downsampled to `clip_fps` and written at
  `clip_fps` (duration ≈ the capped event window, ≤ `clip_max_duration_s`). Raise `clip_fps`
  for smoother (higher-frame-rate) clips.